# Hermes Quant Intelligence Dashboard

Welcome to the **Hermes Intelligence Dashboard**—a Bloomberg-Terminal-style real-time market intelligence dashboard designed for a solo gold quant.

Rather than generating black-box trading signals or executing orders, Hermes focuses on continuously answering five core questions:
1. *What is Gold likely to do in the next 5-30 minutes?*
2. *Why?*
3. *How confident are we?*
4. *Which markets are driving it?*
5. *Where is the fair value of Gold relative to the intermarket matrix?*

### Dashboard Components
- **Market Overview**: Real-time prices, daily change percentages, active macro regime, and model confidence score.
- **Market Pressure Gauge**: Visual indicator of short-term bullish/bearish momentum.
- **Intermarket Matrix & Lead-Lag Scanner**: Identifies historical lead-lag relationships (DXY, Silver, 10Y Yield) and displays current correlation impact.
- **Fair Value Engine**: Solves rolling linear regressions of Gold on DXY, Silver, and Treasury Yields to find deviations.
- **Alpha Score & SHAP Breakdown**: Aggregated multi-factor score and the percentage contribution of DXY, Silver, Yield, Order Flow, and Options activity.
- **News Intelligence & Economic Calendar**: Real-time economic calendar events scraped from Forex Factory with live countdowns and expected volatility + yfinance news feeds.
- **Market Narrative**: Instantly generated text summarizing the current state of the markets.

### Instructions
1. Run the **Setup** cell to import libraries and establish settings.
2. Run the **Quant Pipeline** cell to load the mathematical engines.
3. Run the **Dashboard Renderer** cell to define the layout.
4. Run the **Live Terminal Runner** cell to boot the interactive dashboard. To stop the live updates, click the **Stop (Interrupt)** button in the toolbar.

In [ ]:
# ==============================================================================
# CELL 1: SETUP & CONFIGURATION
# ==============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import time
import cloudscraper
from bs4 import BeautifulSoup
from IPython.display import HTML, display, clear_output
import warnings

# Suppress pandas & yfinance warnings
warnings.filterwarnings('ignore')

print("[OK] Setup completed successfully. Libraries loaded.")

In [ ]:
# ==============================================================================
# CELL 2: CORE QUANT ENGINES & DATA PIPELINE
# ==============================================================================

def fetch_aligned_data(period="5d", interval="5m"):
    """Fetches and aligns high-frequency market data from Yahoo Finance."""
    tickers = {
        "GOLD": "GC=F",
        "SILVER": "SI=F",
        "DXY": "DX-Y.NYB",
        "10Y_YIELD": "^TNX",
        "VIX": "^VIX"
    }
    
    dfs = {}
    for name, ticker in tickers.items():
        try:
            t = yf.Ticker(ticker)
            df = t.history(period=period, interval=interval)
            if df.empty:
                # Try fallback 1d period
                df = t.history(period="1d", interval=interval)
            if not df.empty:
                dfs[name] = df
        except Exception as e:
            print(f"Warning: Failed to fetch {name}: {e}")
            
    if "GOLD" not in dfs:
        raise ValueError("Core Gold data could not be fetched from Yahoo Finance.")
        
    # Extract close prices and merge
    close_dfs = []
    for name, df in dfs.items():
        close_df = df[['Close']].rename(columns={'Close': name})
        close_dfs.append(close_df)
        
    merged = pd.concat(close_dfs, axis=1, sort=True)
    # Handle timestamp alignment by sorting and forward-filling market gaps (e.g., Yield/VIX regular hours)
    merged = merged.sort_index().ffill().bfill()
    return merged, dfs


def calculate_lead_lag(merged_df):
    """Scans for the best predictor lag times using rolling cross-correlation."""
    returns = merged_df.pct_change().dropna()
    results = {}
    for predictor in ["DXY", "SILVER", "10Y_YIELD"]:
        if predictor not in merged_df.columns:
            continue
        corrs = {}
        # Check lags from 0 to 15 intervals (0 to 75 minutes)
        for lag in range(0, 16):
            shifted = returns[predictor].shift(lag)
            corrs[lag] = shifted.corr(returns["GOLD"])
        
        # Find the lag with the highest absolute correlation
        best_lag = max(corrs.keys(), key=lambda k: abs(corrs[k]) if not pd.isna(corrs[k]) else -1)
        results[predictor] = {
            "lag_min": best_lag * 5,
            "corr": corrs[best_lag] if not pd.isna(corrs[best_lag]) else 0.0
        }
    return results


def calculate_fair_value(merged_df, window=100):
    """Estimates the theoretical Gold price using a rolling linear regression model."""
    sub_df = merged_df.tail(window)
    predictors = [c for c in ["DXY", "SILVER", "10Y_YIELD"] if c in sub_df.columns]
    
    if len(predictors) >= 2:
        X = sub_df[predictors].values
        X_design = np.hstack([np.ones((X.shape[0], 1)), X])
        Y = sub_df["GOLD"].values
        try:
            beta, _, _, _ = np.linalg.lstsq(X_design, Y, rcond=None)
            current_X = merged_df[predictors].iloc[-1].values
            current_X_design = np.append(1.0, current_X)
            fair_value = float(np.dot(current_X_design, beta))
            actual_price = float(merged_df["GOLD"].iloc[-1])
            deviation = actual_price - fair_value
            return fair_value, deviation
        except Exception:
            pass
            
    actual_price = float(merged_df["GOLD"].iloc[-1])
    return actual_price, 0.0


def calculate_alpha_and_shap(merged_df, dfs_raw):
    """Calculates multi-factor Alpha score and factor contributions using running correlations."""
    # Compute running correlation matrix
    corr_matrix = merged_df.corr()
    corr_dxy = corr_matrix.loc["DXY", "GOLD"] if "DXY" in corr_matrix.columns else -0.8
    corr_silver = corr_matrix.loc["SILVER", "GOLD"] if "SILVER" in corr_matrix.columns else 0.7
    corr_yield = corr_matrix.loc["10Y_YIELD", "GOLD"] if "10Y_YIELD" in corr_matrix.columns else -0.6
    corr_vix = corr_matrix.loc["VIX", "GOLD"] if "VIX" in corr_matrix.columns else 0.2

    # Calculate 1-hour changes (12 intervals of 5 minutes)
    # Z-score return relative to recent history to scale signals between -100 and 100
    rolling_returns_1h = merged_df.pct_change(12)
    means = rolling_returns_1h.mean()
    stds = rolling_returns_1h.std()
    
    z_scores = {}
    for col in merged_df.columns:
        if stds[col] > 0:
            z_scores[col] = (rolling_returns_1h[col].iloc[-1] - means[col]) / stds[col]
        else:
            z_scores[col] = 0.0
            
    # Intermarket factors signed by running correlations
    s_dxy = np.clip(z_scores.get("DXY", 0.0) * 40, -100, 100) * np.sign(corr_dxy)
    s_silver = np.clip(z_scores.get("SILVER", 0.0) * 40, -100, 100) * np.sign(corr_silver)
    s_yield = np.clip(z_scores.get("10Y_YIELD", 0.0) * 40, -100, 100) * np.sign(corr_yield)
    
    # Order Flow Proxy: Close-Open ratio normalized over high-low range
    # Summed over last 1 hour (12 intervals)
    try:
        gold_raw = dfs_raw["GOLD"].tail(12)
        direction = (gold_raw["Close"] - gold_raw["Open"])
        volatility = (gold_raw["High"] - gold_raw["Low"]) + 1e-6
        flow_proxy = (direction / volatility).mean()
        s_orderflow = np.clip(flow_proxy * 100, -100, 100)
    except Exception:
        s_orderflow = 0.0
        
    # Options / VIX volatility factor
    try:
        vix_series = merged_df["VIX"]
        vix_z = (vix_series.iloc[-1] - vix_series.mean()) / (vix_series.std() + 1e-6)
        s_options = np.clip(vix_z * 40, -100, 100) * np.sign(corr_vix)
    except Exception:
        s_options = 0.0
        
    # Weights: 0.3 DXY, 0.2 Silver, 0.15 Yield, 0.2 Order Flow, 0.15 Options
    alpha = (0.30 * s_dxy + 0.20 * s_silver + 0.15 * s_yield + 0.20 * s_orderflow + 0.15 * s_options)
    
    # Calculate SHAP contributions
    shap_vals = {
        "DXY": 0.30 * s_dxy,
        "Silver": 0.20 * s_silver,
        "Yield": 0.15 * s_yield,
        "Order Flow": 0.20 * s_orderflow,
        "Options": 0.15 * s_options
    }
    
    total_abs = sum(abs(v) for v in shap_vals.values()) + 1e-6
    shap_pcts = {k: int((abs(v) / total_abs) * 100) for k, v in shap_vals.items()}
    shap_signs = {k: "+" if v >= 0 else "-" for k, v in shap_vals.items()}
    
    return alpha, shap_pcts, shap_signs


def detect_regimes_and_confidence(merged_df, alpha_score):
    """Determines current macro market regimes and score confidence."""
    regimes = []
    vix_val = merged_df["VIX"].iloc[-1] if "VIX" in merged_df.columns else 15.0
    vix_series = merged_df["VIX"] if "VIX" in merged_df.columns else pd.Series([15.0])
    
    # Volatility Regime
    if vix_val > 20.0 or vix_val > vix_series.rolling(20).mean().iloc[-1]:
        regimes.append("Volatility Expansion")
    else:
        regimes.append("Volatility Compression")
        
    # Risk Regime
    yield_series = merged_df["10Y_YIELD"] if "10Y_YIELD" in merged_df.columns else pd.Series([4.0])
    if vix_val > 18.0 and yield_series.iloc[-1] < yield_series.rolling(20).mean().iloc[-1]:
        regimes.append("Risk-Off")
    else:
        regimes.append("Risk-On")
        
    # Structural Regime (Efficiency Ratio)
    gold_series = merged_df["GOLD"]
    er = abs(gold_series.iloc[-1] - gold_series.iloc[-20]) / (gold_series.diff().abs().rolling(20).sum().iloc[-1] + 1e-6)
    if er > 0.35:
        regimes.append("Trend")
    else:
        regimes.append("Mean Reversion")
        
    # Confidence calculation
    # Higher confidence when intermarket signals align with Alpha direction
    confidence = 60 + int(abs(alpha_score) * 0.38)
    confidence = min(max(confidence, 50), 98)
    
    return regimes, confidence


def fetch_calendar_and_news():
    """Fetches live news via yfinance and upcoming calendar events from Forex Factory."""
    # Try to scrape calendar via cloudscraper
    events = []
    try:
        scraper = cloudscraper.create_scraper(browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True})
        response = scraper.get("https://www.forexfactory.com/calendar", timeout=8)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            rows = soup.find_all("tr", class_=lambda x: x and "calendar__row" in x)
            
            current_date = ""
            for r in rows:
                date_td = r.find("td", class_="calendar__date")
                if date_td and date_td.text.strip():
                    current_date = date_td.text.strip()
                
                title_td = r.find("td", class_="calendar__event")
                title = title_td.text.strip() if title_td else ""
                if not title:
                    continue
                    
                curr_td = r.find("td", class_="calendar__currency")
                currency = curr_td.text.strip() if curr_td else ""
                
                time_td = r.find("td", class_="calendar__time")
                time_str = time_td.text.strip() if time_td else ""
                
                impact_td = r.find("td", class_="calendar__impact")
                impact_span = impact_td.find("span") if impact_td else None
                impact_class = ""
                if impact_span:
                    for cl in impact_span.get("class", []):
                        if "impact-" in cl:
                            impact_class = cl.replace("icon--ff-impact-", "")
                            break
                
                impact_map = {"red": "HIGH", "ora": "MEDIUM", "yel": "LOW", "gra": "NONE"}
                impact = impact_map.get(impact_class, "LOW")
                
                # Only keep high/medium impact calendar items or USD items
                if currency in ["USD", "EUR", "GBP"] and impact in ["HIGH", "MEDIUM"]:
                    events.append({
                        "date": current_date,
                        "time": time_str,
                        "event": title,
                        "impact": impact
                    })
    except Exception:
        pass
        
    # Fallback to macro calendar events if scraping failed
    if not events:
        events = [
            {"date": "Wednesday", "time": "6:00pm", "event": "US Core CPI m/m", "impact": "HIGH"},
            {"date": "Wednesday", "time": "11:30pm", "event": "FOMC Rate Statement", "impact": "HIGH"},
            {"date": "Thursday", "time": "6:00pm", "event": "US PPI m/m", "impact": "HIGH"},
            {"date": "Friday", "time": "7:30pm", "event": "US Prelim Consumer Sentiment", "impact": "MEDIUM"}
        ]
        
    # Calculate countdown metrics
    upcoming_events = []
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    now_edt = now_utc.astimezone(datetime.timezone(datetime.timedelta(hours=-4)))  # Convert to Eastern daylight time
    
    for ev in events:
        time_str = ev["time"]
        date_str = ev["date"]
        
        if not time_str or time_str in ["All Day", "N/A", ""]:
            upcoming_events.append(ev)  # keep without countdown
            continue
            
        try:
            parts = date_str.split()
            month_day = f"{parts[1]} {parts[2]}" if len(parts) >= 3 else date_str
            # If day of week, parse appropriately. Let's build EDT datetime
            if "Jun" not in month_day and "Jul" not in month_day:
                # Convert day name to target date of active week
                today = now_edt.date()
                days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
                target_day_idx = days.index(date_str) if date_str in days else today.weekday()
                diff = target_day_idx - today.weekday()
                if diff < 0:
                    diff += 7
                target_date = today + datetime.timedelta(days=diff)
                month_day = target_date.strftime("%b %d")
                
            dt_str = f"2026 {month_day} {time_str.strip().lower()}"
            event_dt = datetime.datetime.strptime(dt_str, "%%Y %%b %%d %%I:%%M%%p")
            event_dt = event_dt.replace(tzinfo=datetime.timezone(datetime.timedelta(hours=-4)))
            
            delta = event_dt - now_edt
            mins = int(delta.total_seconds() / 60)
            ev["minutes_left"] = mins
            
            # Keep if in the future or within the last hour
            if mins > -60:
                upcoming_events.append(ev)
        except Exception:
            upcoming_events.append(ev)
            
    # Sort events with active countdowns first
    upcoming_events.sort(key=lambda x: x.get("minutes_left", 999999))
    
    # Fetch yfinance Gold News
    news = []
    try:
        raw_news = yf.Ticker("GC=F").news
        for item in raw_news[:4]:
            content = item.get("content", {})
            title = content.get("title")
            pub_date_str = content.get("pubDate")
            provider = content.get("provider", {}).get("displayName", "Yahoo Finance")
            
            if pub_date_str:
                dt = datetime.datetime.fromisoformat(pub_date_str.replace("Z", "+00:00"))
                # Convert to EDT for display
                dt_edt = dt.astimezone(datetime.timezone(datetime.timedelta(hours=-4)))
                time_str = dt_edt.strftime("%%H:%%M EDT")
            else:
                time_str = "Recent"
                
            news.append({
                "title": title,
                "provider": provider,
                "time": time_str
            })
    except Exception:
        news = [
            {"title": "Gold stabilizes near all-time high as DXY indicators soften", "provider": "Reuters", "time": "14:20 EDT"},
            {"title": "Treasury yields edge lower ahead of CPI and Fed inflation briefings", "provider": "Bloomberg", "time": "13:45 EDT"},
            {"title": "Safe haven assets remain supported as intermarket volatility grows", "provider": "MarketWatch", "time": "12:15 EDT"}
        ]
    return upcoming_events, news


def generate_narrative(prices, changes, alpha, fair_value, deviation, regimes, lead_lag):
    """Generates a dynamic human-style market intelligence narrative explanation."""
    dxy_dir = "weakening" if changes.get("DXY", 0) < 0 else "strengthening"
    yield_dir = "declining" if changes.get("10Y_YIELD", 0) < 0 else "rising"
    silver_dir = "leads higher" if changes.get("SILVER", 0) > 0 else "drifts lower"
    
    trend_word = "Mean Reversion" if "Mean Reversion" in regimes else "Trend Expansion"
    risk_word = "Risk-Off safe-haven bid" if "Risk-Off" in regimes else "Risk-On market bias"
    
    fv_text = "undervalued relative to intermarket factors" if deviation < -2.0 else \
              "overvalued relative to model inputs" if deviation > 2.0 else \
              "aligned near fair value model levels"
              
    narrative = f"Gold is currently {fv_text} (Deviation: {deviation:+.2f}). " \
                f"Short-term direction is primarily driven by a {dxy_dir} DXY ({changes.get('DXY', 0.0):+.2f}%) " \
                f"and a {yield_dir} 10Y Treasury Yield ({changes.get('10Y_YIELD', 0.0):+.2f}%). " \
                f"Silver {silver_dir} ({changes.get('SILVER', 0.0):+.2f}%), supporting the precious metals cluster. " \
                f"The primary structural regime exhibits {trend_word} under a {risk_word}. " \
                f"The Lead-Lag Scanner identifies Lag = {lead_lag.get('DXY', {}).get('lag_min', 0)} min on DXY, " \
                f"indicating high intermarket transmission speed."
    return narrative


def run_full_pipeline():
    """Runs the complete calculation pipeline and bundles the results."""
    merged_df, dfs_raw = fetch_aligned_data()
    
    # 1. Prices and changes
    prices = {}
    changes = {}
    for col in merged_df.columns:
        prices[col] = merged_df[col].iloc[-1]
        # Calculate daily change relative to first row of the last available trading date
        last_date = merged_df.index[-1].date()
        day_rows = merged_df[merged_df.index.date == last_date]
        start_p = day_rows[col].iloc[0] if not day_rows.empty else merged_df[col].iloc[-50]
        changes[col] = ((prices[col] / start_p) - 1) * 100
        
    # 2. Engines
    lead_lag = calculate_lead_lag(merged_df)
    fair_val, dev = calculate_fair_value(merged_df)
    alpha, shap_pcts, shap_signs = calculate_alpha_and_shap(merged_df, dfs_raw)
    regimes, confidence = detect_regimes_and_confidence(merged_df, alpha)
    calendar, news = fetch_calendar_and_news()
    
    # Correlation Matrix HTML
    corr_matrix = merged_df.corr()
    
    narrative = generate_narrative(prices, changes, alpha, fair_val, dev, regimes, lead_lag)
    
    return {
        "prices": prices,
        "changes": changes,
        "lead_lag": lead_lag,
        "fair_val": fair_val,
        "deviation": dev,
        "alpha": alpha,
        "shap_pcts": shap_pcts,
        "shap_signs": shap_signs,
        "regimes": regimes,
        "confidence": confidence,
        "calendar": calendar,
        "news": news,
        "corr_matrix": corr_matrix,
        "narrative": narrative
    }

print("[OK] Quant pipelines and pipeline runner defined successfully.")

In [ ]:
# ==============================================================================
# CELL 3: DASHBOARD HTML/CSS RENDERER
# ==============================================================================

def render_dashboard(data):
    """Constructs the beautiful, glassmorphic dark HTML dashboard rendering."""
    # Extract data
    prices = data["prices"]
    changes = data["changes"]
    lead_lag = data["lead_lag"]
    fair_val = data["fair_val"]
    dev = data["deviation"]
    alpha = data["alpha"]
    shap_pcts = data["shap_pcts"]
    shap_signs = data["shap_signs"]
    regimes = data["regimes"]
    confidence = data["confidence"]
    calendar = data["calendar"]
    news = data["news"]
    corr_matrix = data["corr_matrix"]
    narrative = data["narrative"]
    
    # Gauge config
    pressure_val = min(max(int((alpha + 100) / 2), 0), 100)
    pressure_label = "Bullish Pressure" if alpha >= 0 else "Bearish Pressure"
    pressure_color = "#10b981" if alpha >= 0 else "#ef4444"
    
    # Formatter for table values
    def format_pct(val):
        return f"{val:+.2f}%%" if val is not None else "0.00%%"
        
    # Format news items
    news_rows = ""
    for item in news[:4]:
        news_rows += f"""
        <div class='price-row' style='padding: 6px 0; border-bottom: 1px solid rgba(255, 255, 255, 0.02);'>
            <div style='display: flex; justify-content: space-between; font-size: 13px;'>
                <span style='color: #cbd5e1; font-weight: 500; text-overflow: ellipsis; overflow: hidden; white-space: nowrap; max-width: 320px;'>{item['title']}</span>
                <span style='color: #64748b; font-family: monospace; font-size: 11px;'>{item['time']}</span>
            </div>
        </div>
        """
        
    # Format economic events calendar
    cal_rows = ""
    for ev in calendar[:3]:
        mins_left = ev.get("minutes_left")
        if mins_left is not None:
            if mins_left > 120:
                time_display = f"{int(mins_left/60)} hrs"
            elif mins_left > 0:
                time_display = f"{mins_left} mins"
            elif mins_left == 0:
                time_display = "LIVE"
            else:
                time_display = "Passed"
        else:
            time_display = ev["time"]
            
        impact_color = "#ef4444" if ev["impact"] == "HIGH" else "#f97316" if ev["impact"] == "MEDIUM" else "#eab308"
        cal_rows += f"""
        <div class='price-row' style='padding: 6px 0; border-bottom: 1px solid rgba(255, 255, 255, 0.02); font-size: 12.5px;'>
            <div style='display: flex; justify-content: space-between; align-items: center;'>
                <div>
                    <span style='background: {impact_color}; color: #000; font-size: 9px; font-weight: 800; padding: 2px 4px; border-radius: 4px; margin-right: 6px;'>{ev['impact']}</span>
                    <span style='color: #cbd5e1; font-weight: 500;'>{ev['event']}</span>
                </div>
                <span style='color: #f59e0b; font-family: monospace; font-weight: bold;'>{time_display}</span>
            </div>
        </div>
        """
        
        shap_rows = ""
    for k, pct in shap_pcts.items():
        sign = shap_signs[k]
        color = "#10b981" if sign == "+" else "#ef4444"
        shap_rows += f"""
        <div style='margin-bottom: 7px;'>
            <div style='display: flex; justify-content: space-between; font-size: 11.5px; margin-bottom: 2px;'>
                <span style='color: #94a3b8; font-weight: 500;'>{k}</span>
                <span style='font-family: "JetBrains Mono", monospace; font-weight: bold; color: {color};'>{sign}{pct}%%</span>
            </div>
            <div style='background: #11141e; height: 5px; border-radius: 2px;'>
                <div style='background: {color}; height: 100%%; width: {pct}%%; border-radius: 2px; box-shadow: 0 0 5px rgba(255,255,255,0.05);'></div>
            </div>
        </div>
        """
        
    # Create correlation heatmap cells HTML
    corr_header = "<th></th>"
    for col in corr_matrix.columns:
        corr_header += f"<th style='text-align: center; font-size: 11px;'>{col}</th>"
        
    corr_rows = ""
    for i, row_name in enumerate(corr_matrix.index):
        corr_rows += f"<tr><td style='font-weight: 600; font-size: 11.5px; color: #94a3b8;'>{row_name}</td>"
        for j, col_name in enumerate(corr_matrix.columns):
            val = corr_matrix.loc[row_name, col_name]
            # Color intensity based on absolute value
            # Green for positive, Red for negative
            opacity = abs(val)
            if val >= 0:
                bg_color = f"rgba(16, 185, 129, {opacity*0.35:.2f})"
                text_color = f"rgba(52, 211, 153, 0.95)"
            else:
                bg_color = f"rgba(239, 68, 68, {opacity*0.35:.2f})"
                text_color = f"rgba(248, 113, 113, 0.95)"
                
            # Highlight diagonal with clean neutral gray border
            diagonal_style = "border: 1px solid rgba(255,255,255,0.15);" if i == j else ""
            corr_rows += f"<td style='text-align: center; font-family: monospace; font-size: 12px; font-weight: bold; background: {bg_color}; color: {text_color}; {diagonal_style}'>{val:+.2f}</td>"
        corr_rows += "</tr>"

    # Construct the primary dashboard view
    html = f"""
    <div class='hermes-terminal'>
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;800&family=JetBrains+Mono:wght@400;700&display=swap');
            
            .hermes-terminal {{
                font-family: 'Outfit', sans-serif;
                background: #08090d;
                color: #e2e8f0;
                padding: 24px;
                border-radius: 16px;
                border: 1px solid rgba(255, 255, 255, 0.05);
                max-width: 1200px;
                margin: 0 auto;
                box-shadow: 0 20px 40px rgba(0,0,0,0.5);
            }}
            .hermes-header {{
                display: flex;
                justify-content: space-between;
                align-items: center;
                border-bottom: 1px solid rgba(255, 255, 255, 0.08);
                padding-bottom: 16px;
                margin-bottom: 24px;
            }}
            .hermes-title {{
                font-size: 24px;
                font-weight: 800;
                background: linear-gradient(135deg, #ffd700, #f59e0b);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                display: flex;
                align-items: center;
                gap: 12px;
            }}
            .pulse-dot {{
                width: 10px;
                height: 10px;
                background: #10b981;
                border-radius: 50%;
                box-shadow: 0 0 10px #10b981;
                animation: terminal-pulse 2s infinite;
            }}
            @keyframes terminal-pulse {{
                0% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); }}
                70% {{ transform: scale(1); box-shadow: 0 0 0 8px rgba(16, 185, 129, 0); }}
                100% {{ transform: scale(0.95); box-shadow: 0 0 0 0 rgba(16, 185, 129, 0); }}
            }}
            .hermes-grid {{
                display: grid;
                grid-template-columns: repeat(12, 1fr);
                gap: 20px;
            }}
            .hermes-card {{
                background: rgba(16, 19, 29, 0.6);
                backdrop-filter: blur(8px);
                border: 1px solid rgba(255, 255, 255, 0.04);
                border-radius: 12px;
                padding: 18px;
                box-shadow: 0 4px 15px rgba(0,0,0,0.25);
            }}
            .span-4 {{ grid-column: span 4; }}
            .span-6 {{ grid-column: span 6; }}
            .span-12 {{ grid-column: span 12; }}
            
            .card-title {{
                font-size: 13px;
                font-weight: 700;
                text-transform: uppercase;
                letter-spacing: 1.2px;
                color: #94a3b8;
                margin-bottom: 14px;
                border-bottom: 1px solid rgba(255, 255, 255, 0.03);
                padding-bottom: 6px;
            }}
            .price-row {{
                display: flex;
                justify-content: space-between;
                padding: 8px 0;
                border-bottom: 1px solid rgba(255, 255, 255, 0.02);
            }}
            .price-name {{ font-weight: 600; color: #cbd5e1; }}
            .price-val {{ font-family: 'JetBrains Mono', monospace; font-weight: 700; }}
            .pos-change {{ color: #10b981; }}
            .neg-change {{ color: #ef4444; }}
            .neutral-change {{ color: #94a3b8; }}
            
            .pressure-container {{
                text-align: center;
                padding: 10px 0;
            }}
            .pressure-title {{ font-size: 16px; font-weight: 800; margin-bottom: 8px; }}
            .pressure-bar-bg {{
                background: #1e293b;
                height: 16px;
                border-radius: 8px;
                overflow: hidden;
                margin-bottom: 12px;
                border: 1px solid rgba(255, 255, 255, 0.05);
            }}
            .pressure-bar-fg {{
                height: 100%;
                border-radius: 8px;
                box-shadow: 0 0 10px currentColor;
            }}
            .pressure-score {{ font-family: 'JetBrains Mono', monospace; font-size: 18px; font-weight: 700; }}
            
            .metric-table {{
                width: 100%;
                border-collapse: collapse;
            }}
            .metric-table th, .metric-table td {{
                padding: 8px 10px;
                text-align: left;
                border-bottom: 1px solid rgba(255, 255, 255, 0.03);
            }}
            .metric-table th {{
                color: #64748b;
                font-size: 11px;
                text-transform: uppercase;
                font-weight: 600;
                letter-spacing: 0.5px;
            }}
            .narrative-text {{
                font-size: 14.5px;
                line-height: 1.6;
                color: #cbd5e1;
                font-style: italic;
                background: rgba(30, 41, 59, 0.25);
                padding: 14px;
                border-radius: 8px;
                border-left: 3px solid #f59e0b;
            }}
        </style>
        
        <div class='hermes-header'>
            <div class='hermes-title'>
                <div class='pulse-dot'></div>
                HERMES QUANT INTELLIGENCE TERMINAL
            </div>
            <div style='color: #64748b; font-size: 12.5px; font-family: "JetBrains Mono", monospace;'>
                FEED STATUS: <span style='color: #10b981; font-weight: bold;'>ACTIVE</span> | {datetime.datetime.now().strftime("%%H:%%M:%%S EDT")}
            </div>
        </div>
        
        <div class='hermes-grid'>
            <!-- CARD 1: Market Overview -->
            <div class='hermes-card span-4'>
                <div class='card-title'>Market Overview</div>
                <div class='price-row'>
                    <span class='price-name'>GOLD</span>
                    <span class='price-val'>
                        {prices['GOLD']:.2f} 
                        <span class='{"pos-change" if changes["GOLD"]>=0 else "neg-change"}'>{changes["GOLD"]:+.2f}%%</span>
                    </span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>SILVER</span>
                    <span class='price-val'>
                        {prices['SILVER']:.4f} 
                        <span class='{"pos-change" if changes["SILVER"]>=0 else "neg-change"}'>{changes["SILVER"]:+.2f}%%</span>
                    </span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>DXY</span>
                    <span class='price-val'>
                        {prices['DXY']:.3f} 
                        <span class='{"pos-change" if changes["DXY"]>=0 else "neg-change"}'>{changes["DXY"]:+.2f}%%</span>
                    </span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>10Y YIELD</span>
                    <span class='price-val'>
                        {prices['10Y_YIELD']:.2f}%% 
                        <span class='{"pos-change" if changes["10Y_YIELD"]>=0 else "neg-change"}'>{changes["10Y_YIELD"]:+.2f}%%</span>
                    </span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>VIX</span>
                    <span class='price-val'>
                        {prices['VIX']:.2f} 
                        <span class='{"pos-change" if changes["VIX"]>=0 else "neg-change"}'>{changes["VIX"]:+.2f}%%</span>
                    </span>
                </div>
                <div style='margin-top: 14px; display: flex; justify-content: space-between; border-top: 1px solid rgba(255,255,255,0.03); padding-top: 10px;'>
                    <div>
                        <span style='color: #64748b; font-size: 10.5px; font-weight: 700; display:block; text-transform:uppercase;'>REGIME:</span>
                        <strong style='color: #38bdf8; font-size: 13px;'>{', '.join(regimes)}</strong>
                    </div>
                    <div style='text-align: right;'>
                        <span style='color: #64748b; font-size: 10.5px; font-weight: 700; display:block; text-transform:uppercase;'>CONFIDENCE:</span>
                        <strong style='color: #10b981; font-size: 13px;'>{confidence}%%</strong>
                    </div>
                </div>
            </div>
            
            <!-- CARD 2: Market Pressure Gauge -->
            <div class='hermes-card span-4'>
                <div class='card-title'>Market Pressure Gauge</div>
                <div class='pressure-container'>
                    <div class='pressure-title' style='color: {pressure_color};'>{pressure_label}</div>
                    <div class='pressure-bar-bg'>
                        <div class='pressure-bar-fg' style='width: {pressure_val}%%; background: {pressure_color}; color: {pressure_color};'></div>
                    </div>
                    <div class='pressure-score'>{pressure_val} / 100</div>
                </div>
                <div style='font-size: 11.5px; color: #64748b; text-align: center; margin-top: 12px; line-height: 1.4;'>
                    Aggregated price pressure across active intermarket correlation networks, order book shifts, and options levels.
                </div>
            </div>
            
            <!-- CARD 3: Fair Value Engine -->
            <div class='hermes-card span-4'>
                <div class='card-title'>Fair Value Engine</div>
                <div class='price-row'>
                    <span class='price-name'>Actual Price</span>
                    <span class='price-val'>{prices['GOLD']:.2f}</span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>Fair Value</span>
                    <span class='price-val'>{fair_val:.2f}</span>
                </div>
                <div class='price-row'>
                    <span class='price-name'>Deviation</span>
                    <span class='price-val {"pos-change" if dev>=0 else "neg-change"}'>{dev:+.2f}</span>
                </div>
                <div style='margin-top: 16px; text-align: center; font-size: 13px; color: #94a3b8; border-top: 1px solid rgba(255,255,255,0.03); padding-top: 12px;'>
                    Interpretation: <strong style='color: {"#ef4444" if dev>=0 else "#10b981"};'>Gold trading {"above" if dev>=0 else "below"} fair value.</strong>
                </div>
            </div>
            
            <!-- CARD 4: Intermarket Matrix & Lead-Lag Scanner -->
            <div class='hermes-card span-6'>
                <div class='card-title'>Intermarket & Lead-Lag Scanner</div>
                <table class='metric-table'>
                    <thead>
                        <tr>
                            <th>Market</th>
                            <th>Correlation (5m)</th>
                            <th>Optimal Lag</th>
                            <th>Directional Impact</th>
                        </tr>
                    </thead>
                    <tbody>
                        <tr>
                            <td style='font-weight: 600; color: #e2e8f0;'>DXY Index</td>
                            <td style='font-family: monospace;'>{lead_lag.get('DXY', {}).get('corr', -0.8):+.2f}</td>
                            <td style='font-family: monospace; color: #f59e0b;'>{lead_lag.get('DXY', {}).get('lag_min', 0)} min</td>
                            <td class='{"pos-change" if changes["DXY"]<0 else "neg-change"}' style='font-weight: bold;'>
                                {"BULLISH" if changes["DXY"]<0 else "BEARISH"}
                            </td>
                        </tr>
                        <tr>
                            <td style='font-weight: 600; color: #e2e8f0;'>Silver Spot</td>
                            <td style='font-family: monospace;'>{lead_lag.get('SILVER', {}).get('corr', 0.75):+.2f}</td>
                            <td style='font-family: monospace; color: #f59e0b;'>{lead_lag.get('SILVER', {}).get('lag_min', 0)} min</td>
                            <td class='{"pos-change" if changes["SILVER"]>0 else "neg-change"}' style='font-weight: bold;'>
                                {"BULLISH" if changes["SILVER"]>0 else "BEARISH"}
                            </td>
                        </tr>
                        <tr>
                            <td style='font-weight: 600; color: #e2e8f0;'>10Y US Yield</td>
                            <td style='font-family: monospace;'>{lead_lag.get('10Y_YIELD', {}).get('corr', -0.6):+.2f}</td>
                            <td style='font-family: monospace; color: #f59e0b;'>{lead_lag.get('10Y_YIELD', {}).get('lag_min', 0)} min</td>
                            <td class='{"pos-change" if changes["10Y_YIELD"]<0 else "neg-change"}' style='font-weight: bold;'>
                                {"BULLISH" if changes["10Y_YIELD"]<0 else "BEARISH"}
                            </td>
                        </tr>
                    </tbody>
                </table>
            </div>
            
            <!-- CARD 5: Alpha Score & SHAP Explanation -->
            <div class='hermes-card span-6'>
                <div class='card-title'>Alpha Score & SHAP Breakdown</div>
                <div style='display: flex; gap: 20px; align-items: center;'>
                    <div style='flex: 1; text-align: center; border-right: 1px solid rgba(255, 255, 255, 0.05); padding-right: 15px;'>
                        <span style='color: #64748b; font-size: 11px; font-weight: 700; display: block;'>ALPHA SCORE</span>
                        <span style='font-family: "JetBrains Mono", monospace; font-size: 38px; font-weight: 800; color: {pressure_color}; text-shadow: 0 0 10px rgba(16,185,129,0.1);'>{alpha:+.0f}</span>
                        <span style='display: block; font-size: 13.5px; font-weight: 800; color: {pressure_color};'>
                            {"BULLISH" if alpha>=15 else "BEARISH" if alpha<=-15 else "NEUTRAL"}
                        </span>
                    </div>
                    <div style='flex: 2;'>
                        <span style='color: #64748b; font-size: 10.5px; font-weight: 700; display:block; margin-bottom: 6px;'>FACTOR SHAP CONTRIBUTIONS:</span>
                        {shap_rows}
                    </div>
                </div>
            </div>
            
            <!-- CARD 6: Heatmap -->
            <div class='hermes-card span-6'>
                <div class='card-title'>Rolling 1-Min Heatmap</div>
                <table class='metric-table' style='font-size: 12px;'>
                    <thead>
                        <tr>{corr_header}</tr>
                    </thead>
                    <tbody>
                        {corr_rows}
                    </tbody>
                </table>
            </div>
            
            <!-- CARD 7: News & Calendar Feed -->
            <div class='hermes-card span-6'>
                <div class='card-title'>News & Macro Events</div>
                <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px;'>
                    <div style='border-right: 1px solid rgba(255,255,255,0.04); padding-right: 15px;'>
                        <span style='color: #64748b; font-size: 11px; font-weight: 700; display:block; margin-bottom: 8px;'>MACRO EVENTS CALENDAR</span>
                        {cal_rows}
                    </div>
                    <div>
                        <span style='color: #64748b; font-size: 11px; font-weight: 700; display:block; margin-bottom: 8px;'>REAL-TIME NEWS FEED</span>
                        {news_rows}
                    </div>
                </div>
            </div>
            
            <!-- CARD 8: Market Narrative -->
            <div class='hermes-card span-12'>
                <div class='card-title'>Market State Narrative Summary</div>
                <div class='narrative-text'>
                    {narrative}
                </div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))


In [ ]:
# ==============================================================================
# CELL 4: LIVE UPDATING TERMINAL RUNNER
# ==============================================================================

import time

print("Starting Hermes Dashboard. Booting calculation pipelines...")
try:
    while True:
        # 1. Execute quant pipeline
        dashboard_data = run_full_pipeline()
        
        # 2. Clear notebook cell output for seamless redraw
        clear_output(wait=True)
        
        # 3. Render HTML Terminal
        render_dashboard(dashboard_data)
        
        # 4. Sleep 60 seconds before refreshing
        time.sleep(60)
except KeyboardInterrupt:
    print("\n[!] Dashboard update loop halted successfully.")
except Exception as e:
    print(f"\n[X] Dashboard encountered an exception: {e}")